In [ ]:
!pip -q install unsloth trl transformers datasets peft accelerate "openenv-core[core]"

In [ ]:
!git clone https://github.com/SreeHaran/satellite_scheduler.git
%cd satellite_scheduler

import sys
sys.path.append("/content/satellite_scheduler")

from models import (
    SatelliteSchedulerAction,
    ActionType,
    SatelliteSchedulerObservation,
)
from client import SatelliteSchedulerEnv
from grader import grade_all

!pwd

Cloning into 'satellite_scheduler'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 199 (delta 121), reused 187 (delta 109), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 1.00 MiB | 2.50 MiB/s, done.
Resolving deltas: 100% (121/121), done.
/content/satellite_scheduler
/content/satellite_scheduler


In [ ]:
import os
import json

print(os.path.exists("lora_adapter/adapter_config.json"))
with open("lora_adapter/adapter_config.json", "r") as f:
    cfg = json.load(f)

print(cfg)

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

BASE_MODEL = "unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit"
ADAPTER_PATH = "lora_adapter"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[],
)

model.load_adapter(ADAPTER_PATH, adapter_name="sft_adapter")

print("Model and adapter loaded successfully!")
print("Tokenizer vocab size:", tokenizer.vocab_size)

In [ ]:
SPACE_BASE_URL = "https://sreeharan-satellite-scheduler.hf.space"

In [ ]:
env = SatelliteSchedulerEnv(base_url=SPACE_BASE_URL)
await env.reset()

StepResult(observation=SatelliteSchedulerObservation(done=False, reward=0.0, metadata={}, current_time=0, attitude='sun', busy_status='idle', remaining_action_steps=0, battery_level=90.0, storage_used=0.0, raw_data_amount=0.0, compressed_data_amount=0.0, sunlit_status=True, ground_station_visible=False, pending_request_queue=[], current_selected_request_id=None), reward=0.0, done=False)

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

BASE_MODEL = "unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit"
ADAPTER_PATH = "lora_adapter"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[],
)

model.load_adapter(ADAPTER_PATH, adapter_name="sft_adapter")

print("Model and adapter loaded successfully!")
print("Tokenizer vocab size:", tokenizer.vocab_size)

In [ ]:
import json

OUT_FILE = "grpo_prompts.jsonl"
NUM_EPISODES = 200

def obs_to_prompt(obs):
    return f"""You are a satellite scheduler.

Goal:
1. Complete high priority tasks
2. Use battery and storage efficiently
3. Maximize useful downlink before episode ends

Valid actions:
- {{"action_type":"wait"}}
- {{"action_type":"abort_task"}}
- {{"action_type":"sun_point_for_charging"}}
- {{"action_type":"compress_data"}}
- {{"action_type":"downlink_to_station"}}
- {{"action_type":"capture_image","target_id":"<request_id>"}}

Rules:
- Return exactly one JSON object
- No explanation
- Use target_id only for capture_image

State:
current_time={obs["current_time"]}
attitude={obs["attitude"]}
busy_status={obs["busy_status"]}
remaining_action_steps={obs["remaining_action_steps"]}
battery_level={obs["battery_level"]}
storage_used={obs["storage_used"]}
raw_data_amount={obs["raw_data_amount"]}
compressed_data_amount={obs["compressed_data_amount"]}
sunlit_status={str(obs["sunlit_status"]).lower()}
ground_station_visible={str(obs["ground_station_visible"]).lower()}
pending_request_queue={json.dumps(obs["pending_request_queue"])}
current_selected_request_id={json.dumps(obs["current_selected_request_id"])}
"""

def policy(obs):
    # Replace this with your SFT-policy inference if available.
    if obs["busy_status"] != "idle":
        return {"action_type": "wait"}
    if obs["compressed_data_amount"] > 0 and obs["ground_station_visible"]:
        return {"action_type": "downlink_to_station"}
    if obs["raw_data_amount"] > 0:
        return {"action_type": "compress_data"}
    reqs = obs["pending_request_queue"] or []
    if reqs:
        return {"action_type": "capture_image", "target_id": reqs[0]["id"]}
    if obs["battery_level"] < 25 and obs["sunlit_status"]:
        return {"action_type": "sun_point_for_charging"}
    return {"action_type": "wait"}

def main(env):
    count = 0
    with open(OUT_FILE, "w", encoding="utf-8") as f:
        for _ in range(NUM_EPISODES):
            step_out = env.reset()
            done = False

            while not done:
                obs = step_out["observation"]

                row = {"prompt": obs_to_prompt(obs)}
                f.write(json.dumps(row) + "\n")
                count += 1

                action = policy(obs)
                step_out = env.step(action)
                done = step_out["done"]

    print(f"Saved {count} prompts to {OUT_FILE}")

In [ ]:
import os
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024
TRAIN_FILE = "dataset/sft_training.jsonl"
VAL_FILE = "dataset/sft_validation.jsonl"
OUTPUT_DIR = "outputs"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
val_ds = load_dataset("json", data_files=VAL_FILE, split="train")


def format_row(x):
    return {"text": x["prompt"] + "\n" + x["completion"]}


train_ds = train_ds.map(format_row)
val_ds = val_ds.map(format_row)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=200,
        save_total_limit=2,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        report_to="none",
    ),
)

trainer.train()
model.save_pretrained(os.path.join(OUTPUT_DIR, "lora_adapter"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "lora_adapter"))
print("Done. Saved adapter to", os.path.join(OUTPUT_DIR, "lora_adapter"))


In [ ]:
import json
import os
import matplotlib.pyplot as plt

OUTPUT_DIR = "/content/satellite_scheduler/satellite_scheduler/outputs/lora_adapter"
STATE_FILE = os.path.join(OUTPUT_DIR, "trainer_state.json")

with open(STATE_FILE, "r", encoding="utf-8") as f:
    state = json.load(f)

log_history = state.get("log_history", [])

train_steps = []
train_loss = []

eval_steps = []
eval_loss = []

for row in log_history:
    if "loss" in row and "step" in row:
        train_steps.append(row["step"])
        train_loss.append(row["loss"])
    if "eval_loss" in row and "step" in row:
        eval_steps.append(row["step"])
        eval_loss.append(row["eval_loss"])

plt.figure(figsize=(8, 5))
if train_steps:
    plt.plot(train_steps, train_loss, label="train_loss")
if eval_steps:
    plt.plot(eval_steps, eval_loss, label="eval_loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("SFT Training Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("sft_loss_curve.png", dpi=200)
plt.show()

In [ ]:
async def rollout_with_model(env, max_steps=2):
    history = []
    rewards = []
    actions = []

    result = await env.reset()
    last_reward = 0.01

    for step in range(1, max_steps + 1):
        if result.done:
            break

        obs = result.observation

        decision_text = get_model_decision_local(
            step=step,
            obs=obs,
            last_reward=last_reward,
            history=history,
        )

        action = parse_action_from_text(decision_text)
        result = await env.step(action)

        reward = result.reward or 0.0
        done = result.done

        rewards.append(reward)
        actions.append(
            {
                "step": step,
                "model_text": decision_text,
                "parsed_action": action.action_type.name,
                "reward": reward,
                "done": done,
            }
        )

        history.append(
            f"Step {step}: {action.action_type.name} (reward={reward:+.3f})"
        )
        last_reward = reward

        if done:
            break

    state = await env.state()

    return {
        "actions": actions,
        "rewards": rewards,
        "episode_stats": state.episode_stats,
        "total_reward": sum(rewards),
        "num_steps": len(actions),
    }


async def test_model_rollout():
    env = SatelliteSchedulerEnv(
        base_url=SPACE_BASE_URL,
        connect_timeout_s=60.0,
        message_timeout_s=300.0,
        max_message_size_mb=100.0,
    )
    try:
        await env.connect()
        episode = await rollout_with_model(env, max_steps=2)
        print("num_steps =", episode["num_steps"])
        print("total_reward =", episode["total_reward"])
        for item in episode["actions"]:
            print(item)
    finally:
        await env.close()
        print("closed")


await test_model_rollout()

In [ ]:
import torch
from typing import Any, Dict, List, Optional


def build_chat_messages(user_prompt: str) -> List[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def generate_one_completion(trainer, tokenizer, messages, max_new_tokens=24):
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )

    full_ids = outputs.sequences[0]
    prompt_len = inputs["input_ids"].shape[1]
    completion_ids = full_ids[prompt_len:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    # Approx token logprobs from generation scores
    logprobs = []
    if outputs.scores:
        for tok_id, score_t in zip(completion_ids, outputs.scores):
            lp = torch.log_softmax(score_t[0], dim=-1)[tok_id].item()
            logprobs.append(lp)

    return {
        "prompt_text": prompt_text,
        "prompt_ids": inputs["input_ids"][0].tolist(),
        "completion_text": completion_text if completion_text else "wait",
        "completion_ids": completion_ids.tolist(),
        "logprobs": logprobs,
    }


async def rollout_once(
    trainer,
    env,
    tokenizer,
    dataset_prompt: str = "",
    max_turns: int = 10,
    grade_key: str = "easy",
) -> Dict[str, Any]:
    result = await env.reset()

    history: List[str] = []
    step_rewards: List[float] = []
    parsed_actions: List[str] = []

    all_prompt_ids: List[List[int]] = []
    all_completion_ids: List[List[int]] = []
    all_logprobs: List[List[float]] = []
    raw_text_outputs: List[str] = []

    last_reward = 0.01
    steps_taken = 0

    for step in range(1, max_turns + 1):
        if result.done:
            break

        obs = result.observation

        user_prompt = build_user_prompt(
            step=step,
            obs=obs,
            last_reward=last_reward,
            history=history,
        )

        # optional dataset instruction prefix
        if dataset_prompt:
            user_prompt = f"{dataset_prompt}\n\n{user_prompt}"

        messages = build_chat_messages(user_prompt)

        gen = generate_one_completion(
            trainer=trainer,
            tokenizer=tokenizer,
            messages=messages,
            max_new_tokens=24,
        )

        decision_text = gen["completion_text"]
        action = parse_action_from_text(decision_text)

        result = await env.step(action)

        reward = result.reward or 0.0
        done = result.done

        all_prompt_ids.append(gen["prompt_ids"])
        all_completion_ids.append(gen["completion_ids"])
        all_logprobs.append(gen["logprobs"])
        raw_text_outputs.append(decision_text)

        parsed_actions.append(action.action_type.name)
        step_rewards.append(reward)
        steps_taken = step
        last_reward = reward

        history.append(
            f"Step {step}: {action.action_type.name} (reward={reward:+.3f})"
        )

        if done:
            break

    state = await env.state()
    episode_stats = state.episode_stats
    grades = grade_all(episode_stats)
    score = grades.get(grade_key, 0.0)

    return {
        "prompt_ids": all_prompt_ids,
        "completion_ids": all_completion_ids,
        "logprobs": all_logprobs,
        "raw_outputs": raw_text_outputs,
        "parsed_actions": parsed_actions,
        "step_rewards": step_rewards,
        "total_reward": float(sum(step_rewards)),
        "episode_stats": episode_stats,
        "grades": grades,
        "task_score": score,
        "success": score >= SUCCESS_SCORE_THRESHOLD,
        "steps_taken": steps_taken,
    }

In [ ]:
def rollout_func(prompts, trainer=None):
    episodes = []

    async def _run():
        env = SatelliteSchedulerEnv(base_url=SPACE_BASE_URL)
        await env.connect()
        try:
            for p in prompts:
                ep = await rollout_once(
                    trainer=trainer,
                    env=env,
                    tokenizer=tokenizer,
                    dataset_prompt=p,
                    max_turns=6,
                    grade_key="easy",
                )
                episodes.append(ep)
        finally:
            await env.close()

    asyncio.get_event_loop().run_until_complete(_run())

    return {
        "prompt_ids": [e["prompt_ids"] for e in episodes],
        "completion_ids": [e["completion_ids"] for e in episodes],
        "logprobs": [e["logprobs"] for e in episodes],
        "total_reward": [e["total_reward"] for e in episodes],
        "task_score": [e["task_score"] for e in episodes],
        "success": [e["success"] for e in episodes],
    }